# Multimodal RAG

**Phase 10** · ~60 minutes · Python

**Prerequisites:** See lesson README

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thepandanlabs/applied-ai-from-scratch/blob/main/notebooks/phase-10/07-multimodal-rag.ipynb)

---

> This notebook is auto-generated from the [Applied AI From Scratch](https://github.com/thepandanlabs/applied-ai-from-scratch) curriculum.  
> Run cells top to bottom. Set your API key in the **Setup** cell below.


In [ ]:
!pip install -q Pillow anthropic numpy

import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass  # Running locally — set ANTHROPIC_API_KEY before this cell

print("Setup complete. API key set:", bool(os.environ.get("ANTHROPIC_API_KEY")))

## The Problem

A manufacturing company has 10,000 technical manuals. Each manual has diagrams, assembly illustrations, parts tables, and annotated photographs. The team built a text-only RAG system in Phase 02. It retrieves the right section when an engineer types a text query, but it loses all visual context. An engineer searching "what does the pressure gauge assembly look like?" gets the text surrounding the diagram, not the diagram itself. More critically, two components that look nearly identical have different part numbers. The text says "see Figure 4.3" but Figure 4.3 is not in the context.

The team debates three approaches. Should they store images separately and retrieve by caption? Should they e...

## The Concept

### Three Multimodal RAG Architectures



### Architecture Comparison

| Approach | Index Cost | Query Quality | Infrastructure | Best For |
|----------|------------|---------------|----------------|----------|
| Late Fusion | Low (caption embeddings only) | Medium (depends on caption quality) | Two vector indexes | Large corpora, good existing captions |
| Early Fusion | Medium (VLM call per image) | High (rich descriptions + original images) | One text index + image store | Most production cases |
| Native Multimodal Embedding | Low (one embedding per image) | High for visual similarity quer...

_Read the full lesson narrative in `docs/en.md` or on the course site._

## Implementation

Run each cell in order:

### Setup & Imports

In [ ]:
"""
Multimodal RAG: early fusion implementation.

Indexes documents with image descriptions generated by Claude.
At query time, retrieves text + image context and answers with Claude vision.

Demo mode: creates a tiny mock document set with placeholder images.
No external storage required in demo mode.

Run:
    python main.py --demo          # index + query demo corpus
    python main.py --demo --query "How does the pressure relief valve work?"
"""

import base64
import io
import json
import argparse
import os
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

### Data models

In [ ]:
@dataclass
class Chunk:
    """A text or image chunk from a document, ready for indexing."""
    chunk_id: str
    source: str          # e.g. "manual-A page 3"
    text: str            # text content (or image description for images)
    image_b64: Optional[str] = None  # base64-encoded PNG if this chunk has an image
    embedding: list = field(default_factory=list)

### Synthetic image generator (no external deps beyond Pillow)

In [ ]:
def make_placeholder_image(label: str, width: int = 200, height: int = 150) -> str:
    """
    Generate a simple placeholder PNG with a text label.
    Returns base64-encoded PNG string.
    Requires Pillow. Falls back to a tiny 1x1 PNG if Pillow not available.
    """
    try:
        from PIL import Image, ImageDraw, ImageFont
        img = Image.new("RGB", (width, height), color=(240, 248, 255))
        draw = ImageDraw.Draw(img)
        draw.rectangle([10, 10, width - 10, height - 10], outline=(100, 149, 237), width=2)
        draw.text((width // 2, height // 2), label, fill=(50, 50, 50), anchor="mm")
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return base64.b64encode(buf.getvalue()).decode()
    except ImportError:
        # Minimal valid 1x1 white PNG as fallback
        tiny_png = (
            b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01"
            b"\x00\x00\x00\x01\x08\x02\x00\x00\x00\x90wS\xde\x00\x00"
            b"\x00\x0cIDATx\x9cc\xf8\x0f\x00\x00\x01\x01\x00\x05\x18"
            b"\xd8N\x00\x00\x00\x00IEND\xaeB`\x82"
        )
        return base64.b64encode(tiny_png).decode()

### Demo corpus

In [ ]:
DEMO_CORPUS = [
    {
        "source": "manual-A page 1",
        "text": (
            "Section 3: Pressure Relief Valve Assembly. "
            "The pressure relief valve (PRV) is a critical safety component. "
            "It is located on the upper-left side of the main pump housing. "
            "The valve opens automatically when pressure exceeds 120 PSI. "
            "See Figure 3.1 for the assembly diagram."
        ),
        "has_image": True,
        "image_label": "Fig 3.1\nPRV Assembly",
        "image_description": (
            "Technical diagram showing a cross-section of a pressure relief valve assembly. "
            "The valve body is cylindrical with a spring-loaded poppet on the left. "
            "An adjustment screw at the top controls the opening pressure setpoint. "
            "Two pipe connection ports are labeled: inlet (bottom) and outlet (right side). "
            "A pressure gauge port is visible at the top-right. "
            "Dimensions are annotated: total length 18cm, body diameter 4cm. "
            "A technician would use this diagram to identify component positions during maintenance."
        ),
    },
    {
        "source": "manual-A page 2",
        "text": (
            "Section 4: Pressure Gauge Specifications. "
            "The pressure gauge assembly mounts to the PRV gauge port. "
            "Torque specification: 18-22 Nm using a 24mm wrench. "
            "The gauge dial face shows 0-200 PSI in 10 PSI increments. "
            "Red zone: above 120 PSI (normal operating limit). "
            "See Figure 4.1 for gauge face detail."
        ),
        "has_image": True,
        "image_label": "Fig 4.1\nGauge Face",
        "image_description": (
            "Close-up photograph of a circular pressure gauge dial. "
            "The dial face is white with black numerals 0 through 200 in increments of 10 PSI. "
            "A red arc covers the 120-200 PSI range. "
            "The needle is currently pointing to 85 PSI. "
            "Two indicator marks are at 60 PSI (green dot, normal operating) and 120 PSI (red arrow). "
            "Part number PG-7742-B is printed at the bottom of the dial face."
        ),
    },
    {
        "source": "manual-B page 1",
        "text": (
            "Pump Impeller Maintenance: Monthly Inspection Procedure. "
            "Remove the front cover using the four M8 bolts. "
            "Inspect the impeller for cavitation pitting or blade erosion. "
            "Replace if pitting depth exceeds 1mm. "
            "Apply anti-seize compound to all fastener threads before reassembly."
        ),
        "has_image": True,
        "image_label": "Impeller\nInspection",
        "image_description": (
            "Photograph of a stainless steel centrifugal pump impeller removed from its housing. "
            "Six curved vanes extend radially from a central hub. "
            "One vane on the right side shows visible cavitation pitting - small hemispherical craters "
            "approximately 0.5mm in diameter along the leading edge. "
            "The impeller diameter is 15cm. A ruler is visible in the lower frame for scale. "
            "Part number IMP-3301 is stamped on the hub face."
        ),
    },
    {
        "source": "manual-B page 2",
        "text": (
            "Shaft Seal Replacement. "
            "The mechanical shaft seal prevents fluid leakage along the drive shaft. "
            "Signs of seal failure: visible fluid drip at the pump body, increased vibration. "
            "Replacement interval: every 8,000 operating hours or annually, whichever comes first. "
            "Use only OEM seal kit part number SK-2200-C."
        ),
        "has_image": False,
        "image_label": "",
        "image_description": "",
    },
]

### Embedding (demo: random vectors; production: use text embedding API)

In [ ]:
def demo_embed(text: str, dim: int = 64) -> list:
    """
    Deterministic pseudo-embedding for demo mode.
    In production, replace with openai.embeddings or anthropic embeddings.
    Uses a simple hash-based approach so similar words give similar vectors.
    """
    rng = np.random.default_rng(seed=abs(hash(text[:100])) % (2**32))
    vec = rng.standard_normal(dim)
    # Add keyword-based signal for demo retrieval quality
    keywords = {
        "pressure": [0, 1], "gauge": [0, 1, 2], "valve": [0, 3],
        "impeller": [4, 5], "seal": [6, 7], "pump": [4, 6],
        "assembly": [0, 4], "diagram": [1, 5], "look": [1, 2, 5],
    }
    for kw, dims in keywords.items():
        if kw.lower() in text.lower():
            for d in dims:
                vec[d] += 2.0
    norm = np.linalg.norm(vec)
    return (vec / norm).tolist() if norm > 0 else vec.tolist()

### Indexer

In [ ]:
def build_demo_index(use_real_descriptions: bool = False) -> list:
    """
    Build a multimodal index from the demo corpus.
    use_real_descriptions=True calls Claude for image descriptions.
    use_real_descriptions=False uses pre-written descriptions (no API key needed).
    """
    chunks = []
    for doc in DEMO_CORPUS:
        # Text chunk
        text_chunk = Chunk(
            chunk_id=f"{doc['source']}-text",
            source=doc["source"],
            text=doc["text"],
        )
        text_chunk.embedding = demo_embed(doc["text"])
        chunks.append(text_chunk)

        # Image chunk (if document has an image)
        if doc["has_image"]:
            image_b64 = make_placeholder_image(doc["image_label"])

            if use_real_descriptions:
                # Production path: call Claude for description
                description = describe_image_with_claude(
                    image_b64=image_b64,
                    context_text=doc["text"][:200],
                )
            else:
                # Demo path: use pre-written description
                description = doc["image_description"]

            image_chunk = Chunk(
                chunk_id=f"{doc['source']}-image",
                source=doc["source"],
                text=description,
                image_b64=image_b64,
            )
            image_chunk.embedding = demo_embed(description)
            chunks.append(image_chunk)

    print(f"Index built: {len(chunks)} chunks "
          f"({sum(1 for c in chunks if c.image_b64)} with images)")
    return chunks


def describe_image_with_claude(image_b64: str, context_text: str = "") -> str:
    """
    Call Claude to generate a rich description of an image.
    Used at index time.
    """
    import anthropic
    client = anthropic.Anthropic()
    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=400,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": image_b64,
                        },
                    },
                    {
                        "type": "text",
                        "text": (
                            "Describe this technical diagram in detail for search indexing. "
                            "Include: what is shown, visible component names, spatial relationships, "
                            "numbers or labels, and what a technician would use this diagram for. "
                            + (f"\nSurrounding page text: {context_text}" if context_text else "")
                        ),
                    },
                ],
            }
        ],
    )
    return response.content[0].text

### Retrieval

In [ ]:
def cosine_similarity(a: list, b: list) -> float:
    a_arr, b_arr = np.array(a), np.array(b)
    denom = (np.linalg.norm(a_arr) * np.linalg.norm(b_arr))
    return float(np.dot(a_arr, b_arr) / (denom + 1e-10))


def retrieve(query: str, index: list, top_k: int = 4) -> list:
    """Retrieve top-K chunks by semantic similarity to the query."""
    query_vec = demo_embed(query)
    scored = [
        {**vars(chunk), "score": cosine_similarity(query_vec, chunk.embedding)}
        for chunk in index
    ]
    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]

### Context assembly

In [ ]:
def assemble_context_message(chunks: list) -> list:
    """
    Build the content list for a Claude message from retrieved chunks.
    Interleaves text blocks and image blocks in source order.
    """
    # Sort by source for coherent reading order
    sorted_chunks = sorted(chunks, key=lambda c: c["source"])
    content = []
    for chunk in sorted_chunks:
        content.append({
            "type": "text",
            "text": f"[{chunk['source']}]\n{chunk['text']}"
        })
        if chunk.get("image_b64"):
            content.append({
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/png",
                    "data": chunk["image_b64"],
                }
            })
    return content

### Query (demo: prints assembled context instead of calling Claude)

In [ ]:
def query_demo(query: str, index: list):
    """
    Retrieve and display the assembled multimodal context for a query.
    In demo mode: shows what would be sent to Claude without making an API call.
    """
    print(f"\nQuery: {query}")
    print("-" * 60)

    results = retrieve(query, index, top_k=4)

    print(f"\nTop {len(results)} retrieved chunks:")
    for i, r in enumerate(results, 1):
        has_img = "[+image]" if r.get("image_b64") else ""
        print(f"  {i}. score={r['score']:.3f}  {r['source']} {has_img}")
        print(f"     {r['text'][:100]}...")

    print("\nAssembled context blocks:")
    content = assemble_context_message(results)
    for block in content:
        if block["type"] == "text":
            print(f"  [TEXT]  {block['text'][:80]}...")
        elif block["type"] == "image":
            print(f"  [IMAGE] base64 PNG ({len(block['source']['data'])} chars)")

    print(f"\nContext summary: {len(content)} blocks, "
          f"{sum(1 for b in content if b['type'] == 'image')} images included")


def query_with_claude(query: str, index: list) -> str:
    """
    Full multimodal RAG query using Claude.
    Requires ANTHROPIC_API_KEY environment variable.
    """
    import anthropic
    client = anthropic.Anthropic()

    results = retrieve(query, index, top_k=4)
    context_content = assemble_context_message(results)

    # Add the question at the end
    context_content.append({
        "type": "text",
        "text": f"\nBased on the technical documentation above, answer: {query}"
    })

    response = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=512,
        system=(
            "You are a technical documentation assistant. "
            "Answer questions using only the provided documentation context, "
            "including any diagrams or images shown. "
            "Cite the specific page when referencing visual information."
        ),
        messages=[{"role": "user", "content": context_content}],
    )
    return response.content[0].text

### Main

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Multimodal RAG demo")
    parser.add_argument("--demo", action="store_true",
                        help="Run with synthetic data (no API keys needed)")
    parser.add_argument("--query", type=str,
                        default="What does the pressure gauge assembly look like?",
                        help="Query to run against the index")
    parser.add_argument("--live", action="store_true",
                        help="Call Claude API for real answer (requires ANTHROPIC_API_KEY)")
    args = parser.parse_args()

    if args.demo or not args.live:
        print("\nBuilding multimodal index (demo mode - no API calls)...")
        index = build_demo_index(use_real_descriptions=False)
        query_demo(args.query, index)

        if args.live:
            print("\n--- Live Claude query ---")
            answer = query_with_claude(args.query, index)
            print(f"\nClaude's answer:\n{answer}")
        else:
            print("\nTip: run with --live to call Claude for a real answer.")
            print("Tip: try --query 'How do I inspect the impeller?'")
    else:
        print("Pass --demo to run without API keys.")

### Demo

In [ ]:
main()

---

## Self-Check

Answer these without running code first:

**1. Which multimodal RAG architecture best fits this situation?**

- A. Native multimodal embedding (ColPali) - embeds all 50,000 pages as images, replaces the existing text index
- B. Early fusion - generate VLM descriptions for the 8,000 image pages and add them to the existing text index
- C. Late fusion - build a separate caption index and retrieve from both indexes at query time
- D. Text-only RAG is sufficient - images can be looked up separately

**2. What is the approximate total cost to index the 10,000-image corpus?**

- A. ~$2.50
- B. ~$25
- C. ~$250
- D. ~$2,500

**3. What is the primary concern with the current setup and what should you do?**

- A. No concern - 6,400 tokens for 2 images is well within the 200K window
- B. The context is fine for single queries but will overflow with conversation history
- C. Resize images to a maximum of 1024px on the long edge; 3 images at full resolution can degrade reasoning quality even within window limits
- D. Switch to a text-only retrieval system to avoid token costs

**4. What is the correct behavior of the context assembly step?**

- A. Pass only the text chunks to Claude and provide image links separately
- B. Pass text descriptions of images only - never send actual images
- C. Interleave text blocks and image blocks in source-page order so Claude can reason about both together
- D. Concatenate all text into one block and append all images at the end

**5. How should the team interpret these results?**

- A. Reject multimodal RAG - it hurts text-only query quality
- B. The improvement on visual queries (+33%) significantly outweighs the minor regression on text queries (-3%). Deploy multimodal RAG.
- C. The results are inconclusive - need to re-run on a larger test set before drawing conclusions
- D. Only enable multimodal retrieval for queries that contain the word 'image' or 'diagram'

**6. What is the correct fix?**

- A. Switch to native multimodal embedding (ColPali) - text descriptions cannot capture part numbers
- B. Improve the description generation prompt to explicitly ask for numbers, labels, and part numbers visible in diagrams
- C. Add a separate OCR pipeline to extract text from images and store it alongside the description
- D. Part number queries should not use the RAG system - use a separate database lookup

**7. Which multimodal RAG architecture is most appropriate and why?**

- A. Early fusion - use Claude to describe images, which is more robust than OCR
- B. Late fusion - build a caption index from manually-entered captions
- C. Native multimodal embedding (ColPali) - it embeds page images directly, bypassing unreliable OCR entirely
- D. Text-only RAG after manual re-typing of all documents

_Answers are in `checks.json` in the lesson directory._